# 04 - Split dataset (train / val / test)

Builds the training set from `data/synthetic_degraded/` (+ a slice of undegraded `synthetic_clean/`
so the model still sees some sharp detail) and writes a stratified **70 / 15 / 15** split into
`data/splits/{train,val,test}/{class}/`.

Adapted from Avital's notebook: paths + seed come from `utils.py`; classes are discovered from the
folders on disk. Reproducible (`utils.SEED`).

## Paths

In [ ]:
import sys
from pathlib import Path
# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

import os, random, shutil
random.seed(utils.SEED)

CLEAN_DIR    = utils.CLEAN_DIR
DEGRADED_DIR = utils.DEGRADED_DIR
SPLITS_DIR   = utils.SPLITS_DIR        # data/splits
# Optional labelled real-CCTV test set (per-class subfolders). CLAUDE.md: the real photos are
# calibration/inspiration only - leave this None unless you have a separately-labelled real set.
REAL_DIR = None

TRAIN_RATIO, VAL_RATIO = 0.70, 0.15
CLEAN_FRACTION = 0.30                  # share of each class drawn from the undegraded images
MAX_PER_CLASS  = 4000                  # cap to keep the set manageable
print("clean   :", CLEAN_DIR)
print("degraded:", DEGRADED_DIR)
print("splits  :", SPLITS_DIR)

## Reset the splits folder

In [ ]:
if SPLITS_DIR.exists():
    shutil.rmtree(SPLITS_DIR)
print("reset", SPLITS_DIR)

## Collect images per class

In [ ]:
def collect_images(base):
    data = {}
    if base is None or not Path(base).is_dir():
        return data
    for cls in os.listdir(base):
        cdir = Path(base) / cls
        if cdir.is_dir():
            data[cls] = [str(p) for p in cdir.glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    return data

clean_data    = collect_images(CLEAN_DIR)
degraded_data = collect_images(DEGRADED_DIR)
print({c: len(v) for c, v in degraded_data.items()})

## Build the per-class pool (mostly degraded + some clean)

In [ ]:
data = {}
for cls in degraded_data:
    aug   = degraded_data.get(cls, [])[:]
    clean = clean_data.get(cls, [])[:]
    random.shuffle(aug); random.shuffle(clean)
    n_total = min(MAX_PER_CLASS, len(aug) + len(clean))
    n_clean = int(CLEAN_FRACTION * n_total)
    n_aug   = n_total - n_clean
    selected = aug[:n_aug] + clean[:n_clean]
    random.shuffle(selected)
    data[cls] = selected
print({c: len(v) for c, v in data.items()})

## Stratified split + copy

In [ ]:
from tqdm import tqdm
split_counts = {}
for cls, images in tqdm(data.items(), desc="classes"):
    random.shuffle(images)
    n = len(images); n_tr = int(TRAIN_RATIO * n); n_va = int(VAL_RATIO * n)
    splits = {"train": images[:n_tr], "val": images[n_tr:n_tr+n_va], "test": images[n_tr+n_va:]}
    split_counts[cls] = {}
    for split, paths in splits.items():
        dst_dir = SPLITS_DIR / split / cls
        os.makedirs(dst_dir, exist_ok=True)
        for p in paths:
            shutil.copy(p, dst_dir / os.path.basename(p))
        split_counts[cls][split] = len(paths)

## (Optional) labelled real-CCTV test set

In [ ]:
real_data = collect_images(REAL_DIR)
if real_data:
    REAL_TEST_DIR = utils.DATA_DIR / "real_test"
    if REAL_TEST_DIR.exists():
        shutil.rmtree(REAL_TEST_DIR)
    for cls, imgs in real_data.items():
        os.makedirs(REAL_TEST_DIR / cls, exist_ok=True)
        for p in imgs:
            shutil.copy(p, REAL_TEST_DIR / cls / os.path.basename(p))
    print("real test:", {c: len(v) for c, v in real_data.items()})
else:
    print("No labelled real test set (REAL_DIR is None) - skipping.")

## Summary

In [ ]:
print("Synthetic split:")
for cls, sc in split_counts.items():
    print(f"  {cls}: " + ", ".join(f"{k}={v}" for k, v in sc.items()))